<a href="https://colab.research.google.com/github/Mukhammedgali06/Project/blob/main/World_population.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part A

Section A1 – Web Scraping.

In [ ]:
!pip install requests beautifulsoup4 pandas -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import json

In [ ]:
url = "https://www.worldometers.info/world-population/population-by-country/"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")
table = soup.find("table")
rows = table.find_all("tr")

data = []

for row in rows[1:]:
    cols = row.find_all("td")

    if len(cols) >= 7:
        data.append({
            "Country": cols[1].get_text(strip=True),
            "Population_2023": cols[2].get_text(strip=True),
            "Yearly_Change": cols[3].get_text(strip=True),
            "Net_Change": cols[4].get_text(strip=True),
            "Land_Area": cols[6].get_text(strip=True)
        })

df_web = pd.DataFrame(data)

print("First 5 rows:")
display(df_web.head())

print("\nInfo:")
print(df_web.info())

First 5 rows:


,Country,Population_2023,Yearly_Change,Net_Change,Land_Area
0,India,"1,476,625,576",0.87%,"12,760,051","2,973,190"
1,China,"1,412,914,089",â0.22%,"â3,182,005","9,388,211"
2,United States,"349,035,494",0.51%,"1,759,687","9,147,420"
3,Indonesia,"287,886,782",0.76%,"2,165,546","1,811,570"
4,Pakistan,"259,299,791",1.6%,"4,080,237","770,880"



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233 entries, 0 to 232
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Country          233 non-null    object
 1   Population_2023  233 non-null    object
 2   Yearly_Change    233 non-null    object
 3   Net_Change       233 non-null    object
 4   Land_Area        233 non-null    object
dtypes: object(5)
memory usage: 9.2+ KB
None


Section A2 – API Scraping

In [ ]:
api_url = "https://api.openbrewerydb.org/v1/breweries?per_page=50"

response = requests.get(api_url)
data = response.json()
df_api = pd.json_normalize(data)

df_api = df_api[[
    "id",
    "name",
    "brewery_type",
    "city",
    "state",
    "country"
]]

print("First 5 rows:")
display(df_api.head())
print("\nInfo:")
print(df_api.info())

First 5 rows:


,id,name,brewery_type,city,state,country
0,5128df48-79fc-4f0f-8b52-d06be54d0cec,(405) Brewing Co,micro,Norman,Oklahoma,United States
1,9c5a66c8-cc13-416f-a5d9-0a769c87d318,(512) Brewing Co,micro,Austin,Texas,United States
2,34e8c68b-6146-453f-a4b9-1f6cd99a5ada,1 of Us Brewing Company,micro,Mount Pleasant,Wisconsin,United States
3,6d14b220-8926-4521-8d19-b98a2d6ec3db,10 Barrel Brewing Co,large,Bend,Oregon,United States
4,e2e78bd8-80ff-4a61-a65c-3bfbd9d76ce2,10 Barrel Brewing Co,large,Bend,Oregon,United States



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            50 non-null     object
 1   name          50 non-null     object
 2   brewery_type  50 non-null     object
 3   city          50 non-null     object
 4   state         50 non-null     object
 5   country       50 non-null     object
dtypes: object(6)
memory usage: 2.5+ KB
None


Section A3 – SQLite Data Extraction

In [ ]:
conn = sqlite3.connect('Formula1.sqlite')

query = """
SELECT
    c.circuitId,
    c.circuitRef,
    c.name AS circuit_name,
    c.location,
    c.country,
    c.lat,
    c.lng
FROM circuits c
WHERE c.lat > 0
    AND c.country IN ('USA', 'UK', 'Germany', 'Italy')
ORDER BY c.lng DESC, c.name ASC
LIMIT 100
"""

df_sql = pd.read_sql_query(query, conn)

print("DataFrame result:")
display(df_sql.head())
print("\nDataFrame Info:")
print(df_sql.info())

DataFrame result:


,circuitId,circuitRef,circuit_name,location,country,lat,lng
0,65,pescara,Pescara Circuit,Pescara,Italy,42.4750,14.15080
1,61,avus,AVUS,Berlin,Germany,52.4806,13.25140
2,21,imola,Autodromo Enzo e Dino Ferrari,Imola,Italy,44.3439,11.71670
3,14,monza,Autodromo Nazionale di Monza,Monza,Italy,45.6156,9.28111
4,10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   circuitId     21 non-null     int64  
 1   circuitRef    21 non-null     object 
 2   circuit_name  21 non-null     object 
 3   location      21 non-null     object 
 4   country       21 non-null     object 
 5   lat           21 non-null     float64
 6   lng           21 non-null     float64
dtypes: float64(2), int64(1), object(4)
memory usage: 1.3+ KB
None


In [ ]:
conn1 = sqlite3.connect('Formula1.sqlite')

query1 = """
SELECT
    d.driverId,
    d.driverRef,
    d.number,
    d.code,
    d.forename,
    d.surname,
    d.dob,
    d.nationality
FROM drivers d
WHERE d.nationality IN ('British', 'American', 'German', 'Italian')
  AND d.forename = 'Nick'
ORDER BY d.driverId DESC, d.surname ASC
LIMIT 100
"""

df_sql1 = pd.read_sql_query(query1, conn1)

print("DataFrame result:")
display(df_sql1.head())

print("\nDataFrame Info:")
print(df_sql1.info())

DataFrame result:


,driverId,driverRef,number,code,forename,surname,dob,nationality
0,2,heidfeld,,HEI,Nick,Heidfeld,10/05/1977,German



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   driverId     1 non-null      int64 
 1   driverRef    1 non-null      object
 2   number       1 non-null      object
 3   code         1 non-null      object
 4   forename     1 non-null      object
 5   surname      1 non-null      object
 6   dob          1 non-null      object
 7   nationality  1 non-null      object
dtypes: int64(1), object(7)
memory usage: 196.0+ bytes
None


In [ ]:
conn2 = sqlite3.connect('Formula1.sqlite')

query2 = """
SELECT
    r.raceId,
    r.name AS race_name,
    r.round,
    r.year,
    r.date,
    c.constructorId,
    c.name AS constructor_name,
    c.nationality AS constructor_nationality,
    cr.constructorResultsId,
    cr.points,
    cr.status
FROM constructor_results cr
INNER JOIN races r ON cr.raceId = r.raceId
INNER JOIN constructors c ON cr.constructorId = c.constructorId
WHERE cr.points > 0
    AND r.year >= 2010
ORDER BY cr.points DESC, r.year DESC
LIMIT 100
"""

df_sql2 = pd.read_sql_query(query2, conn2)

print("DataFrame result:")
display(df_sql2.head())

print("\nDataFrame Info:")
print(df_sql2.info())

DataFrame result:


,raceId,race_name,round,year,date,constructorId,constructor_name,constructor_nationality,constructorResultsId,points,status
0,918,Abu Dhabi Grand Prix,19,2014,2014-11-23,3,Williams,British,15006,66,NULL
1,918,Abu Dhabi Grand Prix,19,2014,2014-11-23,131,Mercedes,German,15005,50,NULL
2,974,Monaco Grand Prix,6,2017,2017-05-28,6,Ferrari,Italian,15489,43,NULL
3,979,Hungarian Grand Prix,11,2017,2017-07-30,6,Ferrari,Italian,15540,43,NULL
4,975,Canadian Grand Prix,7,2017,2017-06-11,131,Mercedes,German,15499,43,NULL



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   raceId                   100 non-null    int64 
 1   race_name                100 non-null    object
 2   round                    100 non-null    int64 
 3   year                     100 non-null    int64 
 4   date                     100 non-null    object
 5   constructorId            100 non-null    object
 6   constructor_name         100 non-null    object
 7   constructor_nationality  100 non-null    object
 8   constructorResultsId     100 non-null    int64 
 9   points                   100 non-null    int64 
 10  status                   100 non-null    object
dtypes: int64(5), object(6)
memory usage: 8.7+ KB
None


# Part B – Data Cleaning and Integration

Section B1 – Data Cleaning

In [ ]:
print("Before cleaning df_web:")
display(df_web.head())
print(df_web.info())

Before cleaning df_web:


,Country,Population_2023,Yearly_Change,Net_Change,Land_Area
0,India,"1,476,625,576",0.87%,"12,760,051","2,973,190"
1,China,"1,412,914,089",â0.22%,"â3,182,005","9,388,211"
2,United States,"349,035,494",0.51%,"1,759,687","9,147,420"
3,Indonesia,"287,886,782",0.76%,"2,165,546","1,811,570"
4,Pakistan,"259,299,791",1.6%,"4,080,237","770,880"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233 entries, 0 to 232
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Country          233 non-null    object
 1   Population_2023  233 non-null    object
 2   Yearly_Change    233 non-null    object
 3   Net_Change       233 non-null    object
 4   Land_Area        233 non-null    object
dtypes: object(5)
memory usage: 9.2+ KB
None


In [ ]:
df_web = df_web.drop_duplicates()
df_web = df_web.dropna()

df_web["Population_2023"] = df_web["Population_2023"].astype(str).str.replace(",", "", regex=False)
df_web["Net_Change"] = df_web["Net_Change"].astype(str).str.replace(",", "", regex=False)
df_web["Land_Area"] = df_web["Land_Area"].astype(str).str.replace(",", "", regex=False)
df_web["Yearly_Change"] = df_web["Yearly_Change"].astype(str).str.replace("%", "", regex=False)

df_web["Population_2023"] = pd.to_numeric(df_web["Population_2023"], errors="coerce")
df_web["Net_Change"] = pd.to_numeric(df_web["Net_Change"], errors="coerce")
df_web["Land_Area"] = pd.to_numeric(df_web["Land_Area"], errors="coerce")
df_web["Yearly_Change"] = pd.to_numeric(df_web["Yearly_Change"], errors="coerce")

df_web["Country"] = df_web["Country"].astype(str).str.strip()

print("After cleaning:")
display(df_web.head())
print(df_web.info())

After cleaning:


,Country,Population_2023,Yearly_Change,Net_Change,Land_Area
0,India,1476625576,0.87,12760051.0,2973190
1,China,1412914089,NaN,NaN,9388211
2,United States,349035494,0.51,1759687.0,9147420
3,Indonesia,287886782,0.76,2165546.0,1811570
4,Pakistan,259299791,1.60,4080237.0,770880


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233 entries, 0 to 232
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Country          233 non-null    object 
 1   Population_2023  233 non-null    int64  
 2   Yearly_Change    172 non-null    float64
 3   Net_Change       172 non-null    float64
 4   Land_Area        233 non-null    int64  
dtypes: float64(2), int64(2), object(1)
memory usage: 9.2+ KB
None


In [ ]:
print("Before cleaning df_api:")
display(df_api.head())
print(df_api.info())

Before cleaning df_api:


,id,name,brewery_type,city,state,country
0,5128df48-79fc-4f0f-8b52-d06be54d0cec,(405) Brewing Co,micro,Norman,Oklahoma,United States
1,9c5a66c8-cc13-416f-a5d9-0a769c87d318,(512) Brewing Co,micro,Austin,Texas,United States
2,34e8c68b-6146-453f-a4b9-1f6cd99a5ada,1 of Us Brewing Company,micro,Mount Pleasant,Wisconsin,United States
3,6d14b220-8926-4521-8d19-b98a2d6ec3db,10 Barrel Brewing Co,large,Bend,Oregon,United States
4,e2e78bd8-80ff-4a61-a65c-3bfbd9d76ce2,10 Barrel Brewing Co,large,Bend,Oregon,United States


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            50 non-null     object
 1   name          50 non-null     object
 2   brewery_type  50 non-null     object
 3   city          50 non-null     object
 4   state         50 non-null     object
 5   country       50 non-null     object
dtypes: object(6)
memory usage: 2.5+ KB
None


In [ ]:
df_api = df_api.drop_duplicates()

df_api = df_api.dropna(subset=["country"])

df_api["name"] = df_api["name"].str.strip()
df_api["city"] = df_api["city"].str.strip()
df_api["state"] = df_api["state"].str.strip()

df_api["country"] = df_api["country"].replace({
    "United States": "USA",
    "United States of America": "USA"
})

df_api = df_api[df_api["name"] != ""]
print("After cleaning df_api:")
display(df_api.head())
print(df_api.info())

After cleaning df_api:


,id,name,brewery_type,city,state,country
0,5128df48-79fc-4f0f-8b52-d06be54d0cec,(405) Brewing Co,micro,Norman,Oklahoma,USA
1,9c5a66c8-cc13-416f-a5d9-0a769c87d318,(512) Brewing Co,micro,Austin,Texas,USA
2,34e8c68b-6146-453f-a4b9-1f6cd99a5ada,1 of Us Brewing Company,micro,Mount Pleasant,Wisconsin,USA
3,6d14b220-8926-4521-8d19-b98a2d6ec3db,10 Barrel Brewing Co,large,Bend,Oregon,USA
4,e2e78bd8-80ff-4a61-a65c-3bfbd9d76ce2,10 Barrel Brewing Co,large,Bend,Oregon,USA


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            50 non-null     object
 1   name          50 non-null     object
 2   brewery_type  50 non-null     object
 3   city          50 non-null     object
 4   state         50 non-null     object
 5   country       50 non-null     object
dtypes: object(6)
memory usage: 2.5+ KB
None


In [ ]:
print("Before cleaning df_sql:")
display(df_sql.head())
print(df_sql.info())

Before cleaning df_sql:


,circuitId,circuitRef,circuit_name,location,country,lat,lng
0,65,pescara,Pescara Circuit,Pescara,Italy,42.4750,14.15080
1,61,avus,AVUS,Berlin,Germany,52.4806,13.25140
2,21,imola,Autodromo Enzo e Dino Ferrari,Imola,Italy,44.3439,11.71670
3,14,monza,Autodromo Nazionale di Monza,Monza,Italy,45.6156,9.28111
4,10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   circuitId     21 non-null     int64  
 1   circuitRef    21 non-null     object 
 2   circuit_name  21 non-null     object 
 3   location      21 non-null     object 
 4   country       21 non-null     object 
 5   lat           21 non-null     float64
 6   lng           21 non-null     float64
dtypes: float64(2), int64(1), object(4)
memory usage: 1.3+ KB
None


In [ ]:
df_sql = df_sql.drop_duplicates()

df_sql["lat"] = pd.to_numeric(df_sql["lat"], errors="coerce")
df_sql["lng"] = pd.to_numeric(df_sql["lng"], errors="coerce")

df_sql = df_sql[(df_sql["lat"] >= -90) & (df_sql["lat"] <= 90)]
df_sql = df_sql[(df_sql["lng"] >= -180) & (df_sql["lng"] <= 180)]

df_sql = df_sql.dropna()
print("After cleaning df_sql:")
display(df_sql.head())
print(df_sql.info())

After cleaning df_sql:


,circuitId,circuitRef,circuit_name,location,country,lat,lng
0,65,pescara,Pescara Circuit,Pescara,Italy,42.4750,14.15080
1,61,avus,AVUS,Berlin,Germany,52.4806,13.25140
2,21,imola,Autodromo Enzo e Dino Ferrari,Imola,Italy,44.3439,11.71670
3,14,monza,Autodromo Nazionale di Monza,Monza,Italy,45.6156,9.28111
4,10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   circuitId     21 non-null     int64  
 1   circuitRef    21 non-null     object 
 2   circuit_name  21 non-null     object 
 3   location      21 non-null     object 
 4   country       21 non-null     object 
 5   lat           21 non-null     float64
 6   lng           21 non-null     float64
dtypes: float64(2), int64(1), object(4)
memory usage: 1.3+ KB
None


Section B2 – Kaggle Dataset Integration

In [ ]:
df_kaggle = pd.read_csv('credits.csv').rename(columns={'id': 'movie_id'})

for col in ['cast', 'crew']:
    df_kaggle[col] = df_kaggle[col].fillna('[]').apply(lambda x: eval(x, {"null": None, "None": None}))

df_cast = df_kaggle.explode('cast').reset_index(drop=True)
df_cast = df_cast[['movie_id']].join(pd.json_normalize(df_cast['cast']))

df_crew = df_kaggle.explode('crew').reset_index(drop=True)
df_crew = df_crew[['movie_id']].join(pd.json_normalize(df_crew['crew']))

display(df_cast.head())
display(df_crew.head())


Cast: (564892, 9), Crew: (465085, 8)


,movie_id,cast_id,character,credit_id,gender,id,name,order,profile_path
0,862,14.0,Woody (voice),52fe4284c3a36847f8024f95,2.0,31.0,Tom Hanks,0.0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
1,862,15.0,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,2.0,12898.0,Tim Allen,1.0,/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg
2,862,16.0,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2.0,7167.0,Don Rickles,2.0,/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg
3,862,17.0,Slinky Dog (voice),52fe4284c3a36847f8024fa1,2.0,12899.0,Jim Varney,3.0,/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg
4,862,18.0,Rex (voice),52fe4284c3a36847f8024fa5,2.0,12900.0,Wallace Shawn,4.0,/oGE6JqPP2xH4tNORKNqxbNPYi7u.jpg


,movie_id,credit_id,department,gender,id,job,name,profile_path
0,862,52fe4284c3a36847f8024f49,Directing,2.0,7879.0,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
1,862,52fe4284c3a36847f8024f4f,Writing,2.0,12891.0,Screenplay,Joss Whedon,/dTiVsuaTVTeGmvkhcyJvKp2A5kr.jpg
2,862,52fe4284c3a36847f8024f55,Writing,2.0,7.0,Screenplay,Andrew Stanton,/pvQWsu0qc8JFQhMVJkTHuexUAa1.jpg
3,862,52fe4284c3a36847f8024f5b,Writing,2.0,12892.0,Screenplay,Joel Cohen,/dAubAiZcvKFbboWlj7oXOkZnTSu.jpg
4,862,52fe4284c3a36847f8024f61,Writing,0.0,12893.0,Screenplay,Alec Sokolow,/v79vlRYi94BZUQnkkyznbGUZLjT.jpg


Section B3 – Data Integration Operations

In [ ]:
df_web['Country'] = df_web['Country'].str.strip()
df_sql['country'] = df_sql['country'].replace('UK', 'United Kingdom').str.strip()

df_merged = pd.merge(df_sql, df_web, left_on='country', right_on='Country', how='inner')

print("Merge Result (Routes + Population):")
display(df_merged.head())


Результат Merge (Трассы + Население):


,circuitId,circuitRef,circuit_name,location,country,lat,lng,Country,Population_2023,Yearly_Change,Net_Change,Land_Area
0,65,pescara,Pescara Circuit,Pescara,Italy,42.4750,14.15080,Italy,58926166,NaN,NaN,294140
1,61,avus,AVUS,Berlin,Germany,52.4806,13.25140,Germany,83644258,NaN,NaN,348560
2,21,imola,Autodromo Enzo e Dino Ferrari,Imola,Italy,44.3439,11.71670,Italy,58926166,NaN,NaN,294140
3,14,monza,Autodromo Nazionale di Monza,Monza,Italy,45.6156,9.28111,Italy,58926166,NaN,NaN,294140
4,10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583,Germany,83644258,NaN,NaN,348560


In [ ]:
cols = ['movie_id', 'name', 'credit_id']
df_people = pd.concat([df_cast[cols], df_crew[cols]], axis=0).reset_index(drop=True)
print("Concatenation Result (Actors + Crew):")
display(df_people.head())


Результат Concatenation (Актеры + Команда):


,movie_id,name,credit_id
0,862,Tom Hanks,52fe4284c3a36847f8024f95
1,862,Tim Allen,52fe4284c3a36847f8024f99
2,862,Don Rickles,52fe4284c3a36847f8024f9d
3,862,Jim Varney,52fe4284c3a36847f8024fa1
4,862,Wallace Shawn,52fe4284c3a36847f8024fa5


In [ ]:
df_final = df_people.join(df_cast[['character']], lsuffix='_caller', rsuffix='_other')
display(df_final.head())


,movie_id,name,credit_id,character
0,862,Tom Hanks,52fe4284c3a36847f8024f95,Woody (voice)
1,862,Tim Allen,52fe4284c3a36847f8024f99,Buzz Lightyear (voice)
2,862,Don Rickles,52fe4284c3a36847f8024f9d,Mr. Potato Head (voice)
3,862,Jim Varney,52fe4284c3a36847f8024fa1,Slinky Dog (voice)
4,862,Wallace Shawn,52fe4284c3a36847f8024fa5,Rex (voice)


In [ ]:
print("FINAL DATASET")
display(df_final.head())
df_final.info()


FINAL DATASET


,movie_id,name,credit_id,character
0,862,Tom Hanks,52fe4284c3a36847f8024f95,Woody (voice)
1,862,Tim Allen,52fe4284c3a36847f8024f99,Buzz Lightyear (voice)
2,862,Don Rickles,52fe4284c3a36847f8024f9d,Mr. Potato Head (voice)
3,862,Jim Varney,52fe4284c3a36847f8024fa1,Slinky Dog (voice)
4,862,Wallace Shawn,52fe4284c3a36847f8024fa5,Rex (voice)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1029977 entries, 0 to 1029976
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   movie_id   1029977 non-null  int64 
 1   name       1026788 non-null  object
 2   credit_id  1026788 non-null  object
 3   character  562474 non-null   object
dtypes: int64(1), object(3)
memory usage: 31.4+ MB
